# Variant 18 — поиск на уже обученных OSNet

По аналитике 25.09.2026: сначала разделяем ranking и candidate policy, проверяем пять фиксированных настроек и равновесный ансамбль трёх R1. **Нового обучения нет.** MVP, bbox, validation и старые runs не меняются.

По умолчанию только **inner**: выбор политики на primary, затем frozen-проверка на alternate. Все модели — шаг 800. Открой в прежнем `.venv`: **Restart Kernel → Run All**. Старые notebooks одновременно не запускать.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/retrieval_policy_experiment.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training import retrieval_policy_experiment as experiment
print('Repository:', REPO, '| Torch:', torch.__version__)

## Настройки

`inner` — основной следующий запуск, без outer и calibration. `checkpoints` — отдельная более долгая диагностика всех 11 сохранённых primary-шагов B0/R1; её предложение не меняет final-рецепт.

`final` пока **не запускать**: после разбора inner потребуется отдельное решение и `ALLOW_OUTER_EVALUATION=True`. Тогда пороги будут выбраны только на calibration, сохранены в новых schema-2 bundles и проверены официальным evaluator через CSV. Это оценка на уже просмотренной development validation, не независимый тест.

In [ ]:
PHASE = 'checkpoints'
RUN_NAME = 'policy_v1'
SOURCE_RUN = 'review_v1'
SEED_RUN = 'final_seeds_v1'
ALLOW_OUTER_EVALUATION = False

source_manifest = experiment.review.old.load_json(experiment.review.VARIANT / 'runs' / SOURCE_RUN / 'manifest.json')
torch.set_num_threads(source_manifest['runtime']['torch_threads'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(source_manifest['runtime']['deterministic'], warn_only=True)
print('Phase:', PHASE, '| Encoder training: 0 updates | source device:', source_manifest['runtime']['device'])
print('Ranking grid:', experiment.policy.POLICIES)

## Проверка завершённых запусков

Читаются реальные `selection/confirm_inner` v16 и `results/complete` v17, а не только вывод старого notebook. Проверяются исходные данные, веса, код, старые отчёты и среда. Записывается только новый manifest. При несовпадении хеша не удалять старые manifests и не обходить проверку.

In [ ]:
context = experiment.prepare(RUN_NAME, source_run=SOURCE_RUN, seed_run=SEED_RUN)
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
for name, item in context['confirmation']['selection']['aggregate'].items():
    print(f"Source primary: {name}, step={item['step']}, raw mAP={item['mean']:.5f}")
print('Preflight passed. No encoder inference or training performed by this cell.')

## Расчёт

Печатаются encoder/seed/число изображений, затем готовность каждого draw. Encoder работает только в eval/no-grad. Эмбеддинги и отчёты кешируются с хешами: после прерывания **Restart Kernel → Run All** с теми же настройками.

В inner сначала фиксируется `selection.json`, затем начинается alternate. Там также проверяется same-camera junk. Метрики ансамбля не усредняются с одиночными seed. Отказы на inner не калибруются.

In [ ]:
result = experiment.run(context, phase=PHASE, allow_outer=ALLOW_OUTER_EVALUATION)
print('Encoder training updates:', result['training_updates'])
print('Outer evaluated:', result['outer_evaluated'], '| Promoted:', result['promoted'])
display(Markdown((context['output'] / f'{PHASE.upper()}_RESULTS.md').read_text(encoding='utf-8')))
print('Detailed result:', context['output'] / f'{PHASE}.json')

## После завершения

Сообщи о завершении **inner**; разберём `INNER_RESULTS.md` и `inner.json`. Не выбирать seed/параметры по outer и не запускать final автоматически.

Финальные отчёты отдельно показывают mAP, F1, TNR и **C=0.7F1+0.3TNR**. Кандидатский блок весит 10%, но полный балл без скорости/остальных критериев неизвестен.

K2 CE-scale, EMA/soup, изменение длительности и NiVe в этот эксперимент не включены. Они требуют отдельных сравнений. Новый bundle не переключает web/backend. Официальная скорость и GPU decision parity ещё не измерены.